## Visualizing neural network surfaces

Neural networks tend to be mystified with a lot of pseudoscience.

Fundamentally, you're still drawing a best fit line/surface through a set of points.

Here's what that means, visually.

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt

Generate our dataset

In [2]:
# Making a dataset of concentric spirals for examples sake
# def get_points_in_circle(radius, num_points=100):
#   x_coords = np.arange(-radius, radius, (radius*2/num_points))
#   y_coords = np.sqrt(radius**2 - x_coords**2)

#   x_coords = np.concatenate((x_coords, x_coords))
#   y_coords = np.concatenate((y_coords, -y_coords))

#   label = radius%2 * np.ones(num_points*2)

#   out = np.column_stack((x_coords, y_coords, label))

#   return out


# Just some arbitrary dataset in R3
def min_maxnorm(arr):
  return (arr - np.min(arr))/(np.max(arr) - np.min(arr))
def get_dataset(num_points = 500, noise=0.2):
  # x_coords = np.random.uniform(-10, 10, num_points)
  # y_coords = np.random.uniform(-10, 10, num_points)

  x_coords = np.random.randint(-100, 100, num_points)
  y_coords = np.random.randint(-100, 100, num_points)

  label = x_coords*y_coords
  label = (label - np.mean(label))/(np.std(label))

  label = (1-noise)*label + noise*np.random.randn(num_points)

  out = np.column_stack((x_coords, y_coords, label))

  # shuffle the rows of out
  np.random.shuffle(out)

  return out
dataset = get_dataset()
dataset_df = np.copy(dataset)

# dataset = np.concatenate((get_points_in_circle(1),
#                           get_points_in_circle(2),
#                           get_points_in_circle(3),
#                           get_points_in_circle(4)))

In [3]:
import plotly.figure_factory as ff
import plotly.colors as colors
import plotly.express as px
import plotly.graph_objs as go

# import Layout
from plotly.graph_objs import Layout

import numpy as np
from scipy.spatial import Delaunay

import pandas as pd

def plot_points(x_y_coords):
  x = x_y_coords[:,0]
  y = x_y_coords[:,1]
  label = x_y_coords[:,2]
  plt.scatter(x, y, c=label, cmap='plasma')
  #plt.show()
  return plt


def plot_3d(x_y_coords, title="Connecting those points to see the shape of the dataset as a surface"):
  df = pd.DataFrame(dataset_df, columns=['x', 'y', 'z'])
  zmin = dataset_df[:,2].min()
  zmax = dataset_df[:,2].max()

  scatter_trace = go.Scatter3d(
      x=df['x'],
      y=df['y'],
      z=df['z'],
      mode='markers',
      marker=dict(
        size=3,
        color=df['z'],  # Color by 'z' values
        colorscale='Viridis',
        cmin=zmin,
        cmax=zmax,
        colorbar=dict(title='Color Scale')
      ),
      name='Scatter Plot'
  )

  # Create 3D Tri-Surface Plot
  trisurf_trace = go.Mesh3d(
      x=x_y_coords[:, 0],
      y=x_y_coords[:, 1],
      z=x_y_coords[:, 2],
      opacity=1.0,
      colorscale='Viridis',
      intensity=x_y_coords[:, 2],  # Color by 'z' values
      cmin=zmin,
      cmax=zmax,
      showscale=False,
      flatshading=True,  # Disable smoothing
      name='Tri-Surface Plot'
  )

  # Combine the plots
  fig = go.Figure(data=[scatter_trace, trisurf_trace])

  # Dynamically compute z range based on your predictions

  fig.update_layout(
    scene=dict(
        xaxis_title='X Axis',
        yaxis_title='Y Axis',
        zaxis_title='Z Axis',
        aspectmode='cube',
        zaxis=dict(range=[zmin, zmax])
    ),
    title=title
  )

  return fig

def plot_3d_scatter(x_y_coords, title='3D scatter plot of our dataset, z-axis being the label'):
  df = pd.DataFrame(x_y_coords, columns=['x', 'y', 'z'])

  # change the aspect ratio/scale of axes to be equal
  fig = px.scatter_3d(df, x='x', y='y', z='z', title=title, color='z', color_continuous_scale="Viridis")
  fig.update_traces(marker_size = 3)
  fig.update_layout(scene=dict(aspectmode='cube'))

  # fig.update_layout(scene=dict(xaxis_title='weight', yaxis_title='bias', zaxis_title='loss'))
  # fig.show()
  return fig

def visualize_decisionboundary(xx, yy, predictions):
  # Plot the decision boundary
  zz = predictions.reshape(xx.shape)
  plt.contourf(xx, yy, zz, levels=[0, 0.5, 1], alpha=0.5, colors=['blue', 'yellow'])
  plt.scatter(dataset[:, 0], dataset[:, 1], color=np.where(dataset[:,2]>0.5, 'yellow', 'blue'))

  # Optionally, you can plot the training data points
  # Assuming you have `X_train` and `y_train` as your training data
  # plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, edgecolors='k', cmap='RdYlBu')

  plt.xlabel('X1')
  plt.ylabel('X2')
  plt.title('Decision Boundary')

  return plt

In [4]:
#subplots of all plots
# plot_points(dataset)
# plot_3d(dataset)
plot_3d_scatter(dataset, title="3D Scatter plot of our dataset, we need to find a best fit 'surface' through these points")

## Functions for training the network

In [5]:
# Convert the dataset into torch tensors
import torch.nn as nn
import torch.nn.init as init

dataset = torch.tensor(dataset, dtype=torch.float32)
class LipschitzLinear(nn.Module):
    def __init__(self, in_features, out_features, power_iterations=5, constraint=1.0, batchnorm=False, initmode='uniform'):
        super(LipschitzLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        self.bias = nn.Parameter(torch.zeros(out_features))
        self.u = nn.Parameter(torch.randn(1, out_features), requires_grad=False)
        self.power_iterations = power_iterations
        self.constraint = constraint
        #batchnorm
        if batchnorm:
          self.batchnorm = nn.BatchNorm1d(in_features)
        else:
          self.batchnorm = False
        #weight init
        if initmode == 'normal':
          nn.init.kaiming_normal_(self.weight, a=0.01, mode='fan_in', nonlinearity='leaky_relu')
        elif initmode == 'uniform':
          nn.init.kaiming_uniform_(self.weight, a=0.01, mode='fan_in', nonlinearity='leaky_relu')

    def forward(self, input):
        # Initialize u with a random vector
        out = input
        if self.u is None:
            self.u = nn.Parameter(torch.randn(1, self.out_features), requires_grad=False)

        w = self.weight
        for _ in range(self.power_iterations):
            v = torch.matmul(self.u, w)
            v = v / torch.norm(v)
            u = torch.matmul(v, w.transpose(0, 1))
            u = u / torch.norm(u)

        sigma = torch.matmul(u, torch.matmul(w, v.transpose(0, 1)))
        w = w / sigma
        w = w * self.constraint

        self.u.data.copy_(u)
        if (self.batchnorm!=False):
          out = (self.batchnorm(input)*self.constraint)/torch.max(self.batchnorm.weight)
        out = nn.functional.linear(out, w, self.bias)

        return out

def build_model(in_shape=2, out_shape=1, width_of_each_layer=32, num_hidden_layers=3, lc=False):
    modules = []
    if lc:
      modules = nn.ModuleList([LipschitzLinear(in_shape, width_of_each_layer, constraint=1.0), nn.LeakyReLU(0.01)])
      modules.extend([
          LipschitzLinear(width_of_each_layer, width_of_each_layer, constraint=1.0),
          nn.LeakyReLU(0.01)
          ] * num_hidden_layers)
      modules.extend([LipschitzLinear(width_of_each_layer, out_shape, constraint=1.0)])
    else:
      modules = nn.ModuleList([nn.BatchNorm1d(in_shape),nn.Linear(in_shape, width_of_each_layer), nn.ReLU()])
      modules.extend([
          nn.BatchNorm1d(width_of_each_layer),
          nn.Linear(width_of_each_layer, width_of_each_layer),
          nn.LeakyReLU(0.01)
          ] * num_hidden_layers)
      modules.extend([nn.BatchNorm1d(width_of_each_layer),nn.Linear(width_of_each_layer, out_shape)])

    model = nn.Sequential(*modules)

    return model

# Build a basic torch sequential model
def train_model(model, EPOCHS = 8000, verbose=False):
  # Define the loss function and optimizer
  model.train()
  criterion = torch.nn.MSELoss()

  # With Adam
  optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

  batch_size = 250
  num_batches = dataset.shape[0] // batch_size

  # Train the model
  for epoch in range(EPOCHS):
    acc = 0
    for i in range(num_batches):
      batch = dataset[i*batch_size:(i+1)*batch_size]
      optimizer.zero_grad()
      outputs = model(batch[:, 0:2])
      labels = batch[:, 2]

      acc = torch.sum(torch.where(outputs>0.5, 1, 0) == labels)/batch_size

      loss = criterion(outputs[:,0], labels)
      loss.backward()
      optimizer.step()
    if verbose and (epoch % 500 == 0):
      print(f'Epoch {epoch+1}/{EPOCHS}, Loss: {loss.item():.4f}')
      # print(f'Epoch {epoch+1}/{EPOCHS}, Loss: {loss.item():.4f}, Accuracy: {acc.item():.4f}')

def visualize_model(model, num_layers, width_of_each_layer, on_mesh=True):
  model.eval()

  #get a bunch of random points in a uniform grid
  grid = 0
  if on_mesh:
    x = np.linspace(-100, 100, 100)
    y = np.linspace(-100, 100, 100)
    xx, yy = np.meshgrid(x, y)
    grid = np.column_stack((xx.flatten(), yy.flatten()))
    grid = torch.tensor(grid, dtype=torch.float32).detach()
  else:
    grid = dataset[:,:2]

  predictions = model(grid).detach().numpy().squeeze(1)
  # print(predictions.squeeze(1).shape)
  plot_3d(np.column_stack((grid, predictions)),
        title=f'Network with {num_layers} hidden layer, each {width_of_each_layer} wide, for {3*(width_of_each_layer) + ((num_layers-1)*(width_of_each_layer**2))} parameters').show()

def train_and_visualize(in_shape=2, out_shape=1, width_of_each_layer=32, num_hidden_layers=3, lc=False):
  model = build_model(in_shape, out_shape, width_of_each_layer, num_hidden_layers, lc)
  train_model(model)
  visualize_model(model, num_hidden_layers, width_of_each_layer, True)

In [6]:
train_and_visualize(
    in_shape=2,
    out_shape=1,
    width_of_each_layer=2,
    num_hidden_layers=1
)

In [7]:
train_and_visualize(
    in_shape=2,
    out_shape=1,
    width_of_each_layer=3,
    num_hidden_layers=1
)

In [8]:
train_and_visualize(
    in_shape=2,
    out_shape=1,
    width_of_each_layer=8,
    num_hidden_layers=1
)

In [9]:
train_and_visualize(
    in_shape=2,
    out_shape=1,
    width_of_each_layer=64,
    num_hidden_layers=1
)